# Experiment 1: Train ML Model and Deploy Model to File (PKL)

**Objective:**
- Train a Machine Learning model on the Bank Churn Classification Dataset
- Save the trained model using Pickle (.pkl file)
- Verify that the model loads correctly and produces predictions

**Dataset:** Bank_Churn_Classification_Dataset.csv

**Target Variable:** Churn (Binary Classification - 0: No Churn, 1: Churn)

## Step 1: Install Required Libraries

In [ ]:
import sys
!{sys.executable} -m pip install pandas numpy scikit-learn matplotlib seaborn pickle5 joblib

zsh:1: command not found: pip


## Step 2: Import Libraries

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

ModuleNotFoundError: No module named 'seaborn'

## Step 3: Load and Explore the Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('Bank_Churn_Classification_Dataset.csv', index_col=0)

print(f"Dataset Shape: {df.shape}")
print(f"\nColumn Names:\n{df.columns.tolist()}")
print(f"\nData Types:\n{df.dtypes}")
df.head(10)

In [ ]:
# Basic statistics
print("Dataset Info:")
df.info()
print(f"\nMissing Values:\n{df.isnull().sum()}")
print(f"\nTarget Distribution:\n{df['Churn'].value_counts()}")
print(f"\nTarget Distribution (%):\n{df['Churn'].value_counts(normalize=True) * 100}")

In [ ]:
# Visualize the target distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Churn distribution
sns.countplot(x='Churn', data=df, ax=axes[0], palette='viridis')
axes[0].set_title('Churn Distribution')
axes[0].set_xticklabels(['No Churn (0)', 'Churn (1)'])

# Correlation heatmap for numeric features
numeric_cols = df.select_dtypes(include=[np.number]).columns
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', ax=axes[1], fmt='.2f')
axes[1].set_title('Feature Correlation Heatmap')

plt.tight_layout()
plt.show()

## Step 4: Data Preprocessing

In [ ]:
# Drop CustomerID as it's not a feature
df_processed = df.drop('CustomerID', axis=1)

# Encode categorical variables
label_encoders = {}
categorical_cols = df_processed.select_dtypes(include=['object']).columns
print(f"Categorical columns: {categorical_cols.tolist()}")

for col in categorical_cols:
    le = LabelEncoder()
    df_processed[col] = le.fit_transform(df_processed[col])
    label_encoders[col] = le
    print(f"{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

df_processed.head()

In [ ]:
# Split features and target
X = df_processed.drop('Churn', axis=1)
y = df_processed['Churn']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature columns: {X.columns.tolist()}")

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nFeature scaling completed!")

## Step 5: Train the Model

In [ ]:
# Train a Random Forest Classifier
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_scaled, y_train)
print("Model training completed!")

# Make predictions
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

In [ ]:
# Evaluate the model
print("=" * 50)
print("MODEL EVALUATION")
print("=" * 50)
print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred)}")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No Churn', 'Churn'], yticklabels=['No Churn', 'Churn'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importance, palette='viridis')
plt.title('Feature Importance')
plt.tight_layout()
plt.show()

print(feature_importance)

## Step 6: Save Model Using Pickle (.pkl)

In [ ]:
# Create a directory for all model artifacts
os.makedirs('model_artifacts', exist_ok=True)

# Save the trained model
with open('model_artifacts/churn_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print("Model saved to model_artifacts/churn_model.pkl")

# Save the scaler
with open('model_artifacts/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("Scaler saved to model_artifacts/scaler.pkl")

# Save the label encoders
with open('model_artifacts/label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)
print("Label encoders saved to model_artifacts/label_encoders.pkl")

# Save feature names for later use
with open('model_artifacts/feature_names.pkl', 'wb') as f:
    pickle.dump(X.columns.tolist(), f)
print("Feature names saved to model_artifacts/feature_names.pkl")

# Verify files exist
for fname in os.listdir('model_artifacts'):
    fsize = os.path.getsize(f'model_artifacts/{fname}')
    print(f"  {fname}: {fsize / 1024:.2f} KB")

## Step 7: Verify Model Loading

In [ ]:
# Load the model back from pickle file
with open('model_artifacts/churn_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)
print("Model loaded successfully!")
print(f"Model type: {type(loaded_model)}")

# Load the scaler
with open('model_artifacts/scaler.pkl', 'rb') as f:
    loaded_scaler = pickle.load(f)
print(f"Scaler loaded successfully! Type: {type(loaded_scaler)}")

# Load the label encoders
with open('model_artifacts/label_encoders.pkl', 'rb') as f:
    loaded_encoders = pickle.load(f)
print(f"Label encoders loaded: {list(loaded_encoders.keys())}")

In [ ]:
# Verify predictions match
y_pred_loaded = loaded_model.predict(loaded_scaler.transform(X_test))

# Compare predictions from original and loaded model
predictions_match = np.array_equal(y_pred, y_pred_loaded)
print(f"Predictions from original and loaded model match: {predictions_match}")
print(f"Loaded model accuracy: {accuracy_score(y_test, y_pred_loaded):.4f}")

In [ ]:
# Test with a sample input
sample_input = {
    'Gender': 'Male',
    'SeniorCitizen': 0,
    'Tenure': 30,
    'MonthlyCharges': 70.5,
    'Contract': 'One year',
    'PaymentMethod': 'Electronic check',
    'TotalCharges': 2115.0
}

# Preprocess the sample
sample_df = pd.DataFrame([sample_input])
for col in loaded_encoders:
    if col in sample_df.columns:
        sample_df[col] = loaded_encoders[col].transform(sample_df[col])

sample_scaled = loaded_scaler.transform(sample_df)
prediction = loaded_model.predict(sample_scaled)
prediction_proba = loaded_model.predict_proba(sample_scaled)

print(f"\nSample Input: {sample_input}")
print(f"Prediction: {'Churn' if prediction[0] == 1 else 'No Churn'}")
print(f"Confidence: No Churn={prediction_proba[0][0]:.4f}, Churn={prediction_proba[0][1]:.4f}")
print("\n✅ Model deployment to file verified successfully!")